In [1]:
!python3 -m pip install chromadb

import os
from dotenv import load_dotenv
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_community.vectorstores import Chroma
from langchain_community.retrievers import BM25Retriever
from langchain_classic.retrievers import EnsembleRetriever

  Using cached flatbuffers-25.12.19-py2.py3-none-any.whl.metadata (1.0 kB)
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 21.3/21.3 MB 3.3 MB/s  0:00:06m0:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.0/2.0 MB 1.3 MB/s  0:00:01 eta 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 17.3/17.3 MB 3.1 MB/s  0:00:05m0:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.4/1.4 MB 6.5 MB/s  0:00:00 eta 0:00:01
Using cached flatbuffers-25.12.19-py2.py3-none-any.whl (26 kB)
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 32/32 [chromadb]/32 [chromadb]etry-sdk]onventions]

[notice] A new release of pip is available: 25.3 -> 26.0.1
[notice] To update, run: pip3 install --upgrade pip


/Library/Frameworks/Python.framework/Versions/3.14/lib/python3.14/site-packages/langchain_core/_api/deprecation.py:25: UserWarning: Core Pydantic V1 functionality isn't compatible with Python 3.14 or greater.
  from pydantic.v1.fields import FieldInfo as FieldInfoV1
/Library/Frameworks/Python.framework/Versions/3.14/lib/python3.14/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
load_dotenv()

True

In [3]:
docs = [
    "The employee ID for Jane Doe is EMP-8923.",
    "Human resources handles all payroll inquiries.",
    "If your laptop shows ERR-909, contact IT support immediately.",
    "To reset your password, click the forgot password link."
]

In [4]:
# 2. Setup the Dense Retriever (The Vector Database for Concepts)
embed_model = HuggingFaceEmbeddings(model_name="BAAI/bge-small-en-v1.5")

Loading weights: 100%|██████████| 199/199 [00:00<00:00, 6291.44it/s]
BertModel LOAD REPORT from: BAAI/bge-small-en-v1.5
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [5]:
# Initialize Chroma (This acts as our local Vector Database)
vectorstore = Chroma.from_texts(texts=docs, embedding=embed_model)

# Set it to return the top 2 semantic matches
vector_retriever = vectorstore.as_retriever(search_kwargs={"k":2})

In [6]:
# 3. Setup the Sparse Retriever (The Keyword Search for Exact Matches)
keyword_retriever = BM25Retriever.from_texts(docs)
keyword_retriever.k = 2

In [7]:
# 4. Build the Hybrid Search (Ensemble Retriever)
# We weigh them equally: 50% importance to Exact Keywords, 50% to Meaning
hybrid_retriever = EnsembleRetriever(
    retrievers=[keyword_retriever, vector_retriever],
    weights=[0.5, 0.5]
)

In [8]:
# 5. Let's test the difference!

# Test A: A purely conceptual search
print("--- Concept Search: 'Who do I talk to about my paycheck?' ---")
results_a = hybrid_retriever.invoke("Who do I talk to about my paycheck?")
print(results_a[0].page_content) # Will return the HR payroll document perfectly.

print("\n--- Exact Keyword Search: 'EMP-8923' ---")
# Test B: An exact keyword search
results_b = hybrid_retriever.invoke("EMP-8923")
print(results_b[0].page_content) # Will confidently return Jane Doe's ID document.

--- Concept Search: 'Who do I talk to about my paycheck?' ---
To reset your password, click the forgot password link.

--- Exact Keyword Search: 'EMP-8923' ---
If your laptop shows ERR-909, contact IT support immediately.
